#  CS-245 Machine Learning Project
## Notebook 3 — Modelling, Evaluation & Comparative Analysis
**Project:** Predictive AI for Active Aerodynamics Efficiency and Driver Anomaly Analysis  
**Team:** Aero Intelligence | Hanan Majeed · Maha Mohsin · Anas Norani  
**Dataset:** 2026 F1 Japanese Grand Prix — Red Bull Telemetry (Suzuka Circuit)  
**Course:** CS-245 — Machine Learning | Instructor: Mam Nazia Pervaiz

---
### Notebook Scope — Four-Phase ML Pipeline
| Phase | Model | Type | Purpose |
|-------|-------|------|---------|
| 1 | K-Means Clustering (k=4) | Unsupervised | Track zone identification |
| 2 | Isolation Forest | Unsupervised | Driver anomaly detection |
| 3 | Logistic Regression (L2) | Supervised | Interpretable baseline classifier |
| 4 | Random Forest Classifier | Supervised | Primary predictive engine |

---
> **Prerequisite:** Run `01_Data_Preprocessing.ipynb` first to generate the `artefacts/` directory.

## 0. Setup & Load Preprocessed Artefacts

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ── Scikit-learn models ───────────────────────────────────────────────────────
from sklearn.cluster          import KMeans
from sklearn.ensemble         import IsolationForest, RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.preprocessing    import StandardScaler

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    silhouette_score, ConfusionMatrixDisplay
)

# ── F1 dark theme ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.labelcolor': '#e94560',  'xtick.color': '#ffffff',
    'ytick.color': '#ffffff',      'text.color': '#ffffff',
    'axes.titlecolor': '#e94560',  'axes.edgecolor': '#e94560',
    'grid.color': '#2d2d44',       'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#e94560', 'font.family': 'monospace',
})

F1_RED   = '#e94560'
F1_BLUE  = '#0f3460'
F1_WHITE = '#ffffff'
F1_GOLD  = '#f5a623'
F1_GREEN = '#00d2be'

# ── Load artefacts ────────────────────────────────────────────────────────────
X_train      = np.load('artefacts/X_train.npy')
X_test       = np.load('artefacts/X_test.npy')
y_train      = np.load('artefacts/y_train.npy')
y_test       = np.load('artefacts/y_test.npy')
y_actual_full = np.load('artefacts/y_actual_full.npy')
scaler       = joblib.load('artefacts/standard_scaler.pkl')
FEATURE_COLS = joblib.load('artefacts/feature_cols.pkl')
df           = pd.read_csv('artefacts/df_preprocessed.csv')
df_train     = pd.read_csv('artefacts/df_train.csv')
df_test      = pd.read_csv('artefacts/df_test.csv')

os.makedirs('models', exist_ok=True)


# -- Create output folder for graphs
os.makedirs('graphs', exist_ok=True)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_test: {y_test.shape}')
print(f'Feature columns: {len(FEATURE_COLS)}')
print('  All artefacts loaded.')

---
## Phase 1 — K-Means Clustering: Track Zone Identification

**Goal:** Autonomously identify 4 track zones without human annotation:  
High-Speed Straight · Braking Zone · Corner · Acceleration Zone

In [ ]:
# ── 1.1  Prepare K-Means input features ──────────────────────────────────────
# Input: X, Y, Z (spatial coordinates) + Speed
# These four features define what part of the circuit each telemetry sample belongs to

kmeans_features = ['X', 'Y', 'Z', 'Speed']
X_kmeans_raw    = df[kmeans_features].values

# Scale independently (K-Means is sensitive to scale)
kmeans_scaler   = StandardScaler()
X_kmeans        = kmeans_scaler.fit_transform(X_kmeans_raw)

print('K-Means input shape:', X_kmeans.shape)
print('Features used:', kmeans_features)

In [ ]:
# ── 1.2  Elbow Method — Optimal k selection ───────────────────────────────────
inertias = []
k_range  = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_kmeans)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), inertias, marker='o', color=F1_RED, lw=2)
ax.axvline(4, color=F1_GOLD, lw=1.5, linestyle='--', label='k=4 (selected)')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (Within-Cluster SSE)')
ax.set_title('Elbow Method — K-Means Optimal k Selection')
ax.legend()
plt.tight_layout()
plt.savefig('graphs/kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.3  Fit K-Means (k=4) ────────────────────────────────────────────────────
# k=4 corresponds to the 4 physical track zone types at Suzuka

K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10, max_iter=300)
kmeans.fit(X_kmeans)

df['Track_Zone'] = kmeans.labels_

# Silhouette Score
# Sample for speed (silhouette is O(n²))
sample_idx  = np.random.choice(len(X_kmeans), size=8000, replace=False)
sil_score   = silhouette_score(X_kmeans[sample_idx], kmeans.labels_[sample_idx])

print(f'K-Means (k={K}) fitted successfully.')
print(f'Silhouette Score          : {sil_score:.4f}  (target > 0.60)')
print(f'Inertia                   : {kmeans.inertia_:.2f}')
print(f'\nCluster sizes:')
for c, cnt in zip(*np.unique(kmeans.labels_, return_counts=True)):
    print(f'  Cluster {c}: {cnt:,} samples')

In [ ]:
# ── 1.4  Interpret and label the clusters ────────────────────────────────────
# Use mean speed per cluster to assign human-readable track zone labels

cluster_stats = df.groupby('Track_Zone')['Speed'].agg(['mean', 'min', 'max']).sort_values('mean')
print('Cluster speed statistics (sorted by mean speed):')
print(cluster_stats.round(1))

# Auto-assign labels based on speed ranking
speed_order = cluster_stats.index.tolist()  # Slowest → fastest
zone_labels = {
    speed_order[0]: 'Corner',
    speed_order[1]: 'Braking Zone',
    speed_order[2]: 'Acceleration Zone',
    speed_order[3]: 'High-Speed Straight'
}

df['Track_Zone_Label'] = df['Track_Zone'].map(zone_labels)
print('\nCluster → Track Zone mapping:')
for k, v in zone_labels.items():
    print(f'  Cluster {k}  → {v}')

In [ ]:
# ── 1.5  Visualise track zone map ─────────────────────────────────────────────
zone_colors = {
    'High-Speed Straight': F1_RED,
    'Braking Zone':        F1_GOLD,
    'Acceleration Zone':   F1_GREEN,
    'Corner':              F1_BLUE
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Suzuka Circuit — K-Means Track Zone Segmentation', color=F1_RED, fontsize=13)

# Full circuit
for zone, color in zone_colors.items():
    mask = df['Track_Zone_Label'] == zone
    axes[0].scatter(df.loc[mask, 'X'], df.loc[mask, 'Y'],
                    c=color, s=0.5, alpha=0.6, label=zone, rasterized=True)
axes[0].set_title('Circuit Track Map — Zone Classification')
axes[0].set_xlabel('X coordinate')
axes[0].set_ylabel('Y coordinate')
axes[0].legend(markerscale=8, fontsize=8)

# Zone distribution bar chart
zone_counts = df['Track_Zone_Label'].value_counts()
bars = axes[1].bar(zone_counts.index, zone_counts.values,
                   color=[zone_colors[z] for z in zone_counts.index],
                   edgecolor=F1_WHITE, linewidth=0.5)
axes[1].set_title('Sample Count per Track Zone')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15, labelsize=8)
for bar, val in zip(bars, zone_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 100,
                 f'{val:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('graphs/track_zone_map.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  K-Means complete. Silhouette Score: {sil_score:.4f}')

In [ ]:
# ── 1.6  Append Track_Zone to feature set & re-scale ─────────────────────────
# Track_Zone is a contextual feature for supervised models

FEATURE_COLS_FINAL = FEATURE_COLS + ['Track_Zone']

# Rebuild scaled train/test with Track_Zone appended
df_full_ordered = df.reset_index(drop=True)
split_idx       = len(X_train)

X_train_full = df_full_ordered.iloc[:split_idx][FEATURE_COLS_FINAL].values
X_test_full  = df_full_ordered.iloc[split_idx:][FEATURE_COLS_FINAL].values

scaler_final  = StandardScaler()
X_train_sc    = scaler_final.fit_transform(X_train_full)
X_test_sc     = scaler_final.transform(X_test_full)

print(f'Updated feature set ({len(FEATURE_COLS_FINAL)} features): {FEATURE_COLS_FINAL}')
print(f'X_train_sc shape: {X_train_sc.shape}')
print(f'X_test_sc  shape: {X_test_sc.shape}')

joblib.dump(kmeans, 'models/kmeans_k4.pkl')
print('  K-Means model saved to models/kmeans_k4.pkl')

---
## Phase 2 — Isolation Forest: Driver Anomaly Detection

**Goal:** Detect abnormal driver input patterns and correlate them with aerodynamic timing errors.

In [ ]:
# ── 2.1  Isolation Forest input features ─────────────────────────────────────
# Driver input channels: what the driver physically controls

iso_features = ['Throttle', 'Brake', 'Acceleration', 'Engine_Load',
                'Heavy_Braking', 'Gear_Shift_Active']

X_iso_raw = df[iso_features].values

iso_scaler = StandardScaler()
X_iso      = iso_scaler.fit_transform(X_iso_raw)

print('Isolation Forest input shape:', X_iso.shape)
print('Features used:', iso_features)

In [ ]:
# ── 2.2  Fit Isolation Forest ─────────────────────────────────────────────────
# contamination=0.05 → we expect ~5% of samples to be anomalous driver inputs

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_iso)

df['Anomaly_Flag']  = iso_forest.predict(X_iso)       # -1 = anomaly, +1 = normal
df['Anomaly_Score'] = iso_forest.score_samples(X_iso)  # Lower = more anomalous

n_anomalies    = (df['Anomaly_Flag'] == -1).sum()
anomaly_rate   = n_anomalies / len(df) * 100

print(f'Total samples     : {len(df):,}')
print(f'Anomalies detected: {n_anomalies:,} ({anomaly_rate:.2f}%)')
print(f'Normal samples    : {(df["Anomaly_Flag"]==1).sum():,}')

In [ ]:
# ── 2.3  Correlate anomalies with aero timing errors ─────────────────────────
df['Aero_Deviation']   = (df['Active_Aero_State'] != df['Optimal_Aero']).astype(int)
df['Is_Anomaly']       = (df['Anomaly_Flag'] == -1).astype(int)

# Crosstab: anomaly vs aero deviation
ct = pd.crosstab(df['Is_Anomaly'], df['Aero_Deviation'],
                 rownames=['Is Anomaly'], colnames=['Aero Deviation'])
ct.index = ct.index.map({0: 'Normal', 1: 'Anomaly'})
ct.columns = ct.columns.map({0: 'No Deviation', 1: 'Aero Deviation'})
print('Crosstab — Anomaly vs Aerodynamic Deviation:')
print(ct)

# Of anomalous samples, what % coincide with aero deviations?
anomaly_with_deviation = df[df['Is_Anomaly'] == 1]['Aero_Deviation'].mean() * 100
normal_with_deviation  = df[df['Is_Anomaly'] == 0]['Aero_Deviation'].mean() * 100
print(f'\nOf ANOMALOUS samples → {anomaly_with_deviation:.1f}% have aero timing deviation')
print(f'Of NORMAL samples   → {normal_with_deviation:.1f}% have aero timing deviation')

In [ ]:
# ── 2.4  Anomaly score distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Isolation Forest — Anomaly Detection Results', color=F1_RED, fontsize=12)

# Score distribution
axes[0].hist(df[df['Is_Anomaly']==0]['Anomaly_Score'], bins=60, alpha=0.7,
             color=F1_GREEN, label='Normal', density=True)
axes[0].hist(df[df['Is_Anomaly']==1]['Anomaly_Score'], bins=60, alpha=0.7,
             color=F1_RED, label='Anomaly', density=True)
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Anomaly Score Distribution')
axes[0].legend()

# Anomalies by driver
driver_anomaly = df.groupby('Driver')['Is_Anomaly'].agg(['sum', 'mean'])
driver_anomaly.columns = ['Count', 'Rate']
axes[1].bar(driver_anomaly.index, driver_anomaly['Rate'] * 100,
            color=[F1_RED, F1_GREEN], edgecolor=F1_WHITE)
axes[1].set_ylabel('Anomaly Rate (%)')
axes[1].set_title('Anomaly Rate by Driver')
for i, (drv, row) in enumerate(driver_anomaly.iterrows()):
    axes[1].text(i, row['Rate']*100 + 0.1, f'{row["Count"]:,}\n({row["Rate"]*100:.1f}%)',
                 ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('graphs/isolation_forest_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.5  Anomaly track map — where on circuit do anomalies occur? ─────────────
fig, ax = plt.subplots(figsize=(10, 7))

normal  = df[df['Is_Anomaly'] == 0]
anomaly = df[df['Is_Anomaly'] == 1]

ax.scatter(normal['X'],  normal['Y'],  c=F1_WHITE, s=0.2, alpha=0.2, label='Normal', rasterized=True)
ax.scatter(anomaly['X'], anomaly['Y'], c=F1_RED,   s=2,   alpha=0.7, label='Anomaly', rasterized=True)
ax.set_title('Suzuka Circuit — Anomaly Locations (Red = Driver Anomaly)', fontsize=11)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend(markerscale=5)
plt.tight_layout()
plt.savefig('graphs/anomaly_track_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.6  Root cause feature analysis per anomaly event ───────────────────────
anomaly_df = df[df['Is_Anomaly'] == 1][iso_features + ['Aero_Deviation', 'Driver', 'LapNumber']]
normal_df  = df[df['Is_Anomaly'] == 0][iso_features]

print('Feature means — Anomaly vs Normal samples:')
comparison = pd.DataFrame({
    'Normal Mean':  normal_df.mean().round(3),
    'Anomaly Mean': anomaly_df[iso_features].mean().round(3)
})
comparison['Delta'] = (comparison['Anomaly Mean'] - comparison['Normal Mean']).round(3)
comparison['Primary Cause'] = comparison['Delta'].abs() == comparison['Delta'].abs().max()
print(comparison)

root_cause = comparison['Delta'].abs().idxmax()
print(f'\n  Primary root cause feature: {root_cause}')

joblib.dump(iso_forest, 'models/isolation_forest.pkl')
joblib.dump(iso_scaler, 'models/iso_scaler.pkl')
print('Isolation Forest scaler saved to models/iso_scaler.pkl')
print('  Isolation Forest model saved.')

---
## Phase 3 — Logistic Regression: Interpretable Baseline

**Goal:** Train an interpretable linear classifier; extract feature coefficients.

In [ ]:
# ── 3.1  Train Logistic Regression ───────────────────────────────────────────
# L2 regularisation (default), class_weight balanced to handle any residual imbalance

lr_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

lr_model.fit(X_train_sc, y_train)

y_pred_lr      = lr_model.predict(X_test_sc)
y_prob_lr      = lr_model.predict_proba(X_test_sc)[:, 1]

# Metrics
lr_acc  = accuracy_score(y_test, y_pred_lr)
lr_prec = precision_score(y_test, y_pred_lr)
lr_rec  = recall_score(y_test, y_pred_lr)
lr_f1   = f1_score(y_test, y_pred_lr)
lr_auc  = roc_auc_score(y_test, y_prob_lr)

print('─' * 50)
print('  LOGISTIC REGRESSION — Test Set Results')
print('─' * 50)
print(f'  Accuracy  : {lr_acc:.4f}')
print(f'  Precision : {lr_prec:.4f}')
print(f'  Recall    : {lr_rec:.4f}')
print(f'  F1-Score  : {lr_f1:.4f}')
print(f'  AUC-ROC   : {lr_auc:.4f}')
print('─' * 50)

In [ ]:
# ── 3.2  Detailed classification report ──────────────────────────────────────
print('Classification Report — Logistic Regression:')
print(classification_report(y_test, y_pred_lr,
                             target_names=['Z-Mode (0)', 'X-Mode (1)']))

In [ ]:
# ── 3.3  Confusion Matrix ─────────────────────────────────────────────────────
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                               display_labels=['Z-Mode (0)', 'X-Mode (1)'])
disp.plot(ax=ax, cmap='RdBu_r', colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression', color=F1_RED)
ax.tick_params(colors=F1_WHITE)
plt.tight_layout()
plt.savefig('graphs/cm_logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4  Feature Coefficients (Interpretability) ─────────────────────────────
coef_df = pd.DataFrame({
    'Feature':     FEATURE_COLS_FINAL,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [F1_RED if c > 0 else F1_BLUE for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color=F1_WHITE, lw=0.8)
ax.set_xlabel('Coefficient (L2-Regularised)')
ax.set_title('Logistic Regression Feature Coefficients\n(Positive → favours X-Mode)')
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig('graphs/lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features (by absolute coefficient):')
print(coef_df.head(5).to_string(index=False))

joblib.dump(lr_model, 'models/logistic_regression.pkl')
print('\n  Logistic Regression model saved.')

---
## Phase 4 — Random Forest Classifier: Primary Predictive Engine

**Goal:** Capture non-linear feature interactions; extract feature importance rankings.

In [ ]:
# ── 4.1  Train Random Forest ──────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5
)

rf_model.fit(X_train_sc, y_train)

y_pred_rf = rf_model.predict(X_test_sc)
y_prob_rf = rf_model.predict_proba(X_test_sc)[:, 1]

# Metrics
rf_acc  = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf)
rf_rec  = recall_score(y_test, y_pred_rf)
rf_f1   = f1_score(y_test, y_pred_rf)
rf_auc  = roc_auc_score(y_test, y_prob_rf)

print('─' * 50)
print('  RANDOM FOREST — Test Set Results')
print('─' * 50)
print(f'  Accuracy  : {rf_acc:.4f}')
print(f'  Precision : {rf_prec:.4f}')
print(f'  Recall    : {rf_rec:.4f}')
print(f'  F1-Score  : {rf_f1:.4f}')
print(f'  AUC-ROC   : {rf_auc:.4f}')
print('─' * 50)

In [ ]:
# ── 4.2  Detailed classification report ──────────────────────────────────────
print('Classification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf,
                             target_names=['Z-Mode (0)', 'X-Mode (1)']))

In [ ]:
# ── 4.3  Confusion Matrix ─────────────────────────────────────────────────────
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['Z-Mode (0)', 'X-Mode (1)'])
disp.plot(ax=ax, cmap='RdBu_r', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', color=F1_RED)
ax.tick_params(colors=F1_WHITE)
plt.tight_layout()
plt.savefig('graphs/cm_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.4  Feature Importances (MDI) ───────────────────────────────────────────
fi_df = pd.DataFrame({
    'Feature':    FEATURE_COLS_FINAL,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1], color=F1_RED, edgecolor='none')
ax.set_xlabel('Mean Decrease in Impurity (Importance)')
ax.set_title('Random Forest — Feature Importances')
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig('graphs/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 8 features by importance:')
print(fi_df.head(8).to_string(index=False))

joblib.dump(rf_model, 'models/random_forest.pkl')
print('\n  Random Forest model saved.')

---
## 5. Comparative ROC Curve Analysis

In [ ]:
# ── ROC curves — both supervised models on the same plot ─────────────────────
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_lr, tpr_lr, color=F1_GOLD,  lw=2, label=f'Logistic Regression (AUC = {lr_auc:.4f})')
ax.plot(fpr_rf, tpr_rf, color=F1_RED,   lw=2, label=f'Random Forest (AUC = {rf_auc:.4f})')
ax.plot([0, 1], [0, 1],  color=F1_WHITE, lw=1, linestyle='--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — Logistic Regression vs Random Forest')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('graphs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Full Model Comparative Analysis

In [ ]:
# ── Comprehensive metrics comparison table ────────────────────────────────────
comparison_table = pd.DataFrame({
    'Model': [
        'K-Means (k=4)',
        'Isolation Forest',
        'Logistic Regression (L2)',
        'Random Forest'
    ],
    'Type': [
        'Unsupervised (Clustering)',
        'Unsupervised (Anomaly)',
        'Supervised (Classification)',
        'Supervised (Classification)'
    ],
    'Primary Metric': [
        f'Silhouette Score: {sil_score:.4f}',
        f'Anomaly Rate: {anomaly_rate:.2f}%',
        f'F1-Score: {lr_f1:.4f}',
        f'F1-Score: {rf_f1:.4f}'
    ],
    'Accuracy': ['-', '-', f'{lr_acc:.4f}', f'{rf_acc:.4f}'],
    'Precision': ['-', '-', f'{lr_prec:.4f}', f'{rf_prec:.4f}'],
    'Recall': ['-', '-', f'{lr_rec:.4f}', f'{rf_rec:.4f}'],
    'F1-Score': ['-', '-', f'{lr_f1:.4f}', f'{rf_f1:.4f}'],
    'AUC-ROC': ['-', '-', f'{lr_auc:.4f}', f'{rf_auc:.4f}'],
    'Target Met': [
        '' if sil_score >= 0.60 else '',
        ' (5% rate expected)',
        '' if lr_f1 >= 0.90 else '',
        '' if rf_f1 >= 0.90 else ''
    ]
})

print('FULL MODEL COMPARISON TABLE:')
pd.set_option('display.max_colwidth', 35)
comparison_table.set_index('Model')

In [ ]:
# ── Visual metrics comparison ─────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
lr_vals = [lr_acc, lr_prec, lr_rec, lr_f1, lr_auc]
rf_vals = [rf_acc, rf_prec, rf_rec, rf_f1, rf_auc]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, lr_vals, width, label='Logistic Regression',
               color=F1_GOLD, edgecolor=F1_WHITE, linewidth=0.5)
bars2 = ax.bar(x + width/2, rf_vals, width, label='Random Forest',
               color=F1_RED, edgecolor=F1_WHITE, linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Supervised Model Comparison — All Evaluation Metrics')
ax.axhline(0.90, color=F1_GREEN, lw=1, linestyle='--', alpha=0.7, label='Target threshold (0.90)')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('graphs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Driver Deviation Analysis (Post-Training)

In [ ]:
# ── Compare RF model predictions vs driver's actual aero state ────────────────
# On the TEST set only

df_test_analysis = df_test.copy().reset_index(drop=True)
df_test_analysis['RF_Prediction']   = y_pred_rf
df_test_analysis['Driver_Actual']   = df_test_analysis['Active_Aero_State'].astype(int)
df_test_analysis['Timing_Deviation'] = (
    df_test_analysis['RF_Prediction'] != df_test_analysis['Driver_Actual']
).astype(int)

# Deviation per driver per lap
deviation_lap = df_test_analysis.groupby(['Driver', 'LapNumber'])['Timing_Deviation'].agg(
    total_deviations='sum',
    deviation_rate='mean'
).reset_index()

print('Per-Driver Timing Deviation Summary (Test Set):')
print(df_test_analysis.groupby('Driver')['Timing_Deviation'].agg(
    ['sum', 'mean']
).rename(columns={'sum': 'Total Deviations', 'mean': 'Deviation Rate'}).round(4))

In [ ]:
# ── Visualise deviations across test laps ────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

for driver, color in [('VER', F1_RED), ('HAD', F1_GREEN)]:
    drv_data = deviation_lap[deviation_lap['Driver'] == driver]
    if not drv_data.empty:
        ax.bar(drv_data['LapNumber'] + (0.2 if driver == 'VER' else -0.2),
               drv_data['total_deviations'], width=0.4,
               color=color, alpha=0.8, label=driver)

ax.set_xlabel('Lap Number')
ax.set_ylabel('Aero Timing Deviations')
ax.set_title('AI-Predicted vs Driver-Actual Aero Deviations per Lap (Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig('graphs/deviation_per_lap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Save All Artefacts & Final Summary Report

In [ ]:
# ── Save anomaly-enriched full dataframe ─────────────────────────────────────
df.to_csv('artefacts/df_final_with_anomalies.csv', index=False)

# Save comparison table
comparison_table.to_csv('artefacts/model_comparison_table.csv', index=False)

print('  All model artefacts saved:')
for f in sorted(os.listdir('models')):
    print(f'   models/{f}')

In [ ]:
# ── FINAL COMPREHENSIVE REPORT ────────────────────────────────────────────────
best_model = 'Random Forest' if rf_f1 >= lr_f1 else 'Logistic Regression'
best_f1    = max(rf_f1, lr_f1)

print('=' * 68)
print('  CS-245 ML PROJECT — FINAL RESULTS SUMMARY')
print('  Team Aero Intelligence | NUST SEECS')
print('=' * 68)
print()
print('  PHASE 1 — K-Means Clustering')
print(f'    Clusters (k)         : 4 (Corner | Braking | Accel | Straight)')
print(f'    Silhouette Score     : {sil_score:.4f}  (target > 0.60)')
print(f'    Status               : {" TARGET MET" if sil_score >= 0.60 else " Below target"}')
print()
print('  PHASE 2 — Isolation Forest')
print(f'    Anomalies Detected   : {n_anomalies:,} ({anomaly_rate:.2f}% of dataset)')
print(f'    Anomalies + Deviation: {anomaly_with_deviation:.1f}% correlation with timing errors')
print(f'    Primary Root Cause   : {root_cause}')
print(f'    Status               :  ANOMALY LOG GENERATED')
print()
print('  PHASE 3 — Logistic Regression (Baseline)')
print(f'    Accuracy  : {lr_acc:.4f}')
print(f'    Precision : {lr_prec:.4f}')
print(f'    Recall    : {lr_rec:.4f}')
print(f'    F1-Score  : {lr_f1:.4f}  (target > 0.90)')
print(f'    AUC-ROC   : {lr_auc:.4f}')
print(f'    Status    : {" TARGET MET" if lr_f1 >= 0.90 else " Below target"}')
print()
print('  PHASE 4 — Random Forest (Primary)')
print(f'    Accuracy  : {rf_acc:.4f}')
print(f'    Precision : {rf_prec:.4f}')
print(f'    Recall    : {rf_rec:.4f}')
print(f'    F1-Score  : {rf_f1:.4f}  (target > 0.90)')
print(f'    AUC-ROC   : {rf_auc:.4f}')
print(f'    Status    : {" TARGET MET" if rf_f1 >= 0.90 else " Below target"}')
print()
print('  WINNER')
print(f'    Best Model  : {best_model}')
print(f'    Best F1     : {best_f1:.4f}')
print()
print('  CS-245 REQUIREMENTS CHECK')
print('     2 Supervised models   : Logistic Regression + Random Forest')
print('     1+ Unsupervised models: K-Means + Isolation Forest (exceeded)')
print('     Comparative analysis  : Full metrics table + ROC curves')
print('     Visualisations        : 15+ plots generated and saved')
print('     Physics-derived target: Optimal_Aero (not driver-actual)')
print('=' * 68)